Import libraries

In [50]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from cfrnet import CFRnet
import sys
from pathlib import Path
project_root = Path("/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [51]:
%time train_df = pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/val_men.csv")

CPU times: user 22.4 ms, sys: 6 ms, total: 28.4 ms
Wall time: 28 ms
CPU times: user 13.1 ms, sys: 3 ms, total: 16.1 ms
Wall time: 16.1 ms
CPU times: user 4.59 ms, sys: 1 ms, total: 5.59 ms
Wall time: 5.6 ms


In [52]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['conversion']
treatment_feature = ['treatment']

In [53]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [54]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [55]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [56]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [57]:
epochs = 150
alpha = 1
lr = 1e-3
wd = 8e-6
method = "mmd_rbf"
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 20
early_stop_start = 0
shared_hidden = 200
outcome_hidden = 100
activation = torch.nn.ReLU

print (f" epochs = {epochs}")
print (f" method = {method}")
print (f" alpha = {alpha}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f"activation = {activation}")

 epochs = 150
 method = mmd_rbf
 alpha = 1
 learning rate = 0.001
 weight decay = 8e-06
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 20
 shared hidden = 200
 outcome hidden = 100
activation = <class 'torch.nn.modules.activation.ReLU'>


In [ ]:
import pandas as pd
import numpy as np
import torch
from itertools import product

# 1. Grid search config
seeds = [412312, 42, 1874, 902745, 1]
lr_grid = [3e-5, 5e-5, 1e-4, 3e-4, 5e-4, 1e-3]
wd_grid = [0.0, 1e-6, 1e-5, 1e-4]
method_grid = ["mmd_linear", "mmd_rbf"]
alpha_grid = [0.1, 0.5, 1.0]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
grid_results = []

total_cfg = len(lr_grid) * len(wd_grid) * len(method_grid) * len(alpha_grid)
print(f"🔄 Bắt đầu grid search: {total_cfg} cấu hình")

# 2. Loop over all hyperparameter combinations
for grid_lr, grid_wd, grid_method, grid_alpha in product(lr_grid, wd_grid, method_grid, alpha_grid):
    run_rows = []

    print("\n" + "=" * 110)
    print(
        f"Cấu hình: lr={grid_lr:.1e}, weight_decay={grid_wd:.1e}, "
        f"method={grid_method}, alpha={grid_alpha}"
    )
    print("=" * 110)

    for SEED in seeds:
        seed_everything(SEED)

        # Reinitialize model for each seed
        cfr = CFRnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            alpha=grid_alpha,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            method=grid_method,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_dropout=outcome_dropout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        cfr.fit(train_loader, val_loader)

        # Validation prediction
        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p = cfr.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)

        # Test prediction
        x_test_device = x_men_test_t.to(device)
        y0_pred, y1_pred = cfr.predict(x_test_device)

        uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
        y_true = y_men_test_t.detach().cpu().numpy().flatten()
        t_true = t_men_test_t.detach().cpu().numpy().flatten()

        # ATE error
        ate_pred = uplift_pred.mean()
        ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

        row = {
            "Seed": SEED,
            "lr": grid_lr,
            "weight_decay": grid_wd,
            "method": grid_method,
            "alpha": grid_alpha,
            "Val_AUQC": current_val_auqc,
            "AUUC": auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
            "AUQC": auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
            "Lift": lift(y_true, t_true, uplift_pred, h=0.3),
            "KRCC": krcc(y_true, t_true, uplift_pred, bins=100),
            "ATE_Err": abs(ate_pred - ate_true)
        }
        run_rows.append(row)

        print(
            f"  ✔ Seed {SEED} | Val_AUQC={current_val_auqc:.4f} | "
            f"Test_AUQC={row['AUQC']:.4f}"
        )

    # Aggregate each hyperparameter combination
    df_pair = pd.DataFrame(run_rows)
    mean_pair = df_pair[["Val_AUQC", "AUUC", "AUQC", "Lift", "KRCC", "ATE_Err"]].mean()
    std_pair = df_pair[["Val_AUQC", "AUUC", "AUQC", "Lift", "KRCC", "ATE_Err"]].std()

    grid_results.append({
        "lr": grid_lr,
        "weight_decay": grid_wd,
        "method": grid_method,
        "alpha": grid_alpha,
        "mean_Val_AUQC": mean_pair["Val_AUQC"],
        "mean_AUUC": mean_pair["AUUC"],
        "mean_AUQC": mean_pair["AUQC"],
        "mean_Lift": mean_pair["Lift"],
        "mean_KRCC": mean_pair["KRCC"],
        "mean_ATE_Err": mean_pair["ATE_Err"],
        "std_Val_AUQC": std_pair["Val_AUQC"],
        "std_AUUC": std_pair["AUUC"],
        "std_AUQC": std_pair["AUQC"],
        "std_Lift": std_pair["Lift"],
        "std_KRCC": std_pair["KRCC"],
        "std_ATE_Err": std_pair["ATE_Err"]
    })

# 3. Final ranking by mean test AUQC
df_grid = pd.DataFrame(grid_results).sort_values("mean_AUQC", ascending=False).reset_index(drop=True)
best_cfg = df_grid.iloc[0]

print("\n" + "=" * 130)
print(f"{'KẾT QUẢ GRID SEARCH (XẾP HẠNG THEO MEAN TEST AUQC)':^130}")
print("=" * 130)

display_cols = [
    "lr", "weight_decay", "method", "alpha",
    "mean_Val_AUQC", "mean_AUUC", "mean_AUQC", "mean_Lift", "mean_KRCC", "mean_ATE_Err",
    "std_Val_AUQC", "std_AUUC", "std_AUQC", "std_Lift", "std_KRCC", "std_ATE_Err"
]

print(df_grid[display_cols].to_string(
    index=False,
    formatters={
        "lr": "{:.1e}".format,
        "weight_decay": "{:.1e}".format,
        "alpha": "{:.1f}".format,
        "mean_Val_AUQC": "{:.4f}".format,
        "mean_AUUC": "{:.4f}".format,
        "mean_AUQC": "{:.4f}".format,
        "mean_Lift": "{:.4f}".format,
        "mean_KRCC": "{:.4f}".format,
        "mean_ATE_Err": "{:.4f}".format,
        "std_Val_AUQC": "{:.4f}".format,
        "std_AUUC": "{:.4f}".format,
        "std_AUQC": "{:.4f}".format,
        "std_Lift": "{:.4f}".format,
        "std_KRCC": "{:.4f}".format,
        "std_ATE_Err": "{:.4f}".format
    }
))

print("-" * 130)
print("Best hyperparameters by mean TEST AUQC:")
print(
    f"  lr={best_cfg['lr']:.1e}, weight_decay={best_cfg['weight_decay']:.1e}, "
    f"method={best_cfg['method']}, alpha={best_cfg['alpha']:.1f}"
)
print(f"  mean TEST AUQC = {best_cfg['mean_AUQC']:.4f} ± {best_cfg['std_AUQC']:.4f}")
print("=" * 130)

In [58]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy (Chỉ xử lý, không in kết quả lẻ)
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    cfrnet = CFRnet(
        input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=lr, 
        alpha=alpha, method=method,
        weight_decay=wd, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
        shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
        early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
        activation=activation
    )
    
    cfrnet.fit(train_loader, val_loader)
    
    # Predict
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = cfrnet.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

print("\n" + "="*85)
print(f"{'CHI TIẾT TỪNG SEED':^85}")
print("="*85)
# Sử dụng to_string để in bảng đẹp mắt
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format, 'AUQC': '{:,.4f}'.format, 
    'Lift': '{:,.4f}'.format, 'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*85)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^85}")
print("-" * 85)
summary_data = []
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*85)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 412312
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 1
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini


Epoch 1/150 | Loss: -7.5308 | Val Loss: -7.5456 | Val Qini: 0.7701 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7701 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -7.6173 | Val Loss: -7.6575 | Val Qini: 0.6969 (patience: 1/20)EMA Trend: 0.7591 | (patience: 1/20)
Epoch 3/150 | Loss: -7.8319 | Val Loss: -7.8905 | Val Qini: 0.2383 (patience: 2/20)EMA Trend: 0.6810 | (patience: 2/20)
Epoch 4/150 | Loss: -7.9765 | Val Loss: -7.9811 | Val Qini: 0.3568 (patience: 3/20)EMA Trend: 0.6324 | (patience: 3/20)
Epoch 5/150 | Loss: -7.9827 | Val Loss: -7.9819 | Val Qini: 0.3538 (patience: 4/20)EMA Trend: 0.5906 | (patience: 4/20)
Epoch 6/150 | Loss: -7.9773 | Val Loss: -7.9818 | Val Qini: 0.3538 (patience: 5/20)EMA Trend: 0.5551 | (patience: 5/20)
Epoch 7/150 | Loss: -7.9805 | Val Loss: -7.9818 | Val Qini: 0.3510 (patience: 6/20)EMA Trend: 0.5245 | (patience: 6/20)
Epoch 8/150 | Loss: -7.9833 | Val Loss: -7.9818 | Val Qini: 0.3483 (patience: 7/20)EMA Trend: 0.4980 | (patience: 7/20)
Epoch 9/150 | Loss: -7

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Conversion/CFRnet/cfrnet.py:270: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: -7.5261 | Val Loss: -7.5503 | Val Qini: 0.3249 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3249 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -7.6777 | Val Loss: -7.7438 | Val Qini: 0.2788 (patience: 1/20)EMA Trend: 0.3180 | (patience: 1/20)
Epoch 3/150 | Loss: -7.9333 | Val Loss: -7.9631 | Val Qini: 0.7399 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3813 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: -7.9836 | Val Loss: -7.9819 | Val Qini: 0.7322 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.4339 | ✓ above trend but not peak (patience: 1/20)
Epoch 5/150 | Loss: -7.9793 | Val Loss: -7.9819 | Val Qini: 0.7263 ✓ above trend but not peak (patience: 2/20)EMA Trend: 0.4778 | ✓ above trend but not peak (patience: 2/20)
Epoch 6/150 | Loss: -7.9872 | Val Loss: -7.9819 | Val Qini: 0.7204 ✓ above trend but not peak (patience: 3/20)EMA Trend: 0.5142 | ✓ above trend but not peak (patience: 3/20)
Epoch 7/150 | Loss: -7.9829 | Val Loss: -7.9819 | Val Qini: 0.7147 ✓ above trend b

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Conversion/CFRnet/cfrnet.py:270: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


...

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Conversion/CFRnet/cfrnet.py:270: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: -7.5541 | Val Loss: -7.5891 | Val Qini: 0.7271 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7271 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -7.7575 | Val Loss: -7.8298 | Val Qini: 0.7309 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7276 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: -7.9683 | Val Loss: -7.9798 | Val Qini: 0.2811 (patience: 1/20)EMA Trend: 0.6607 | (patience: 1/20)
Epoch 4/150 | Loss: -7.9830 | Val Loss: -7.9818 | Val Qini: 0.2834 (patience: 2/20)EMA Trend: 0.6041 | (patience: 2/20)
Epoch 5/150 | Loss: -7.9830 | Val Loss: -7.9817 | Val Qini: 0.2854 (patience: 3/20)EMA Trend: 0.5563 | (patience: 3/20)
Epoch 6/150 | Loss: -7.9845 | Val Loss: -7.9817 | Val Qini: 0.2907 (patience: 4/20)EMA Trend: 0.5164 | (patience: 4/20)
Epoch 7/150 | Loss: -7.9797 | Val Loss: -7.9817 | Val Qini: 0.2947 (patience: 5/20)EMA Trend: 0.4832 | (patience: 5/20)
Epoch 8/150 | Loss: -7.9815 | Val Loss: -7.9817 | Val Qini: 0.2949 (patience: 6/20)EMA Trend: 0.4549 | (patience: 6/20)
Epoc

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Conversion/CFRnet/cfrnet.py:270: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: -7.5391 | Val Loss: -7.5645 | Val Qini: 0.6282 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.6282 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -7.6978 | Val Loss: -7.7651 | Val Qini: 0.1849 (patience: 1/20)EMA Trend: 0.5617 | (patience: 1/20)
Epoch 3/150 | Loss: -7.9543 | Val Loss: -7.9741 | Val Qini: 0.5596 (patience: 2/20)EMA Trend: 0.5614 | (patience: 2/20)
Epoch 4/150 | Loss: -7.9796 | Val Loss: -7.9819 | Val Qini: 0.7232 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5857 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: -7.9812 | Val Loss: -7.9818 | Val Qini: 0.7319 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.6076 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: -7.9769 | Val Loss: -7.9818 | Val Qini: 0.7308 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.6261 | ✓ above trend but not peak (patience: 1/20)
Epoch 7/150 | Loss: -7.9800 | Val Loss: -7.9818 | Val Qini: 0.7252 ✓ above trend but not peak (patience: 2/20)EMA Trend: 0.6409 | ✓ above trend but not peak (patience: 2/20

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Conversion/CFRnet/cfrnet.py:270: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
